# Day 15 setup

Run this cell first. It loads the config and prepares logging and manifest paths for the later tasks.

In [2]:
import json
import logging
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
import yaml

project_folder = Path.cwd()
if not (project_folder / "data").is_dir():
    project_folder = project_folder.parent

day15_folder = project_folder / "day-15"
if str(day15_folder) not in sys.path:
    sys.path.insert(0, str(day15_folder))

from pipeline_helpers import (
    MissingRequiredColumnError,
    apply_column_mapping,
    check_required_columns,
    find_missing_fields,
    get_required_columns,
    get_unprocessed_files,
    make_bad_record,
    save_manifest,
)

config_file = day15_folder / "day15_config.yaml"
with config_file.open(encoding="utf-8") as file:
    config = yaml.safe_load(file)

log_folder = project_folder / config["log_folder"]
log_folder.mkdir(exist_ok=True)
log_file = log_folder / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True,
)

manifest_file = project_folder / config["manifest_file"]
if manifest_file.is_file():
    with manifest_file.open(encoding="utf-8") as file:
        processed_files = json.load(file)
else:
    processed_files = []

data_folder = project_folder / "data"
csv_files = sorted(data_folder.glob("*.csv"))
files_to_process = get_unprocessed_files(csv_files, processed_files)
for file in csv_files:
    if file.name in processed_files:
        logging.info("Skipping already processed file: %s", file.name)

logging.info("Starting Olist data loading")

2026-11-25 15:12:12,897 INFO: Skipping already processed file: olist_customers_dataset.csv
2026-11-25 15:12:12,899 INFO: Skipping already processed file: olist_customers_semicolon.csv
2026-11-25 15:12:12,902 INFO: Skipping already processed file: olist_geolocation_dataset.csv
2026-11-25 15:12:12,903 INFO: Skipping already processed file: olist_order_items_dataset.csv
2026-11-25 15:12:12,904 INFO: Skipping already processed file: olist_order_payments_dataset.csv
2026-11-25 15:12:12,906 INFO: Skipping already processed file: olist_order_reviews_dataset.csv
2026-11-25 15:12:12,907 INFO: Skipping already processed file: olist_orders_dataset.csv
2026-11-25 15:12:12,910 INFO: Skipping already processed file: olist_products_dataset.csv
2026-11-25 15:12:12,911 INFO: Skipping already processed file: olist_sellers_dataset.csv
2026-11-25 15:12:12,912 INFO: Skipping already processed file: product_category_name_translation.csv
2026-11-25 15:12:12,913 INFO: Starting Olist data loading


## 2. Read all Olist CSV files

The notebook finds the project folder automatically, whether Jupyter starts in the project folder or `day-15`.

In [3]:
dataframes = {}
for file in files_to_process:
    separator = ";" if "semicolon" in file.name else ","
    dataframes[file.stem] = pd.read_csv(file, sep=separator)

    print(f"{file.name}: {dataframes[file.stem].shape[0]} rows, {dataframes[file.stem].shape[1]} columns")
    logging.info("Loaded %s", file.name)

print(f"\nLoaded {len(dataframes)} new CSV files.")
logging.info("Finished loading %s new CSV files", len(dataframes))

2026-11-25 15:12:15,501 INFO: Finished loading 0 new CSV files



Loaded 0 new CSV files.


## 3. Use a config file to map column names

Read the column names from `day15_config.yaml` and rename matching columns in each loaded DataFrame.

In [4]:
column_mapping = config["column_mapping"]
mapped_dataframes = {}

for name, dataframe in dataframes.items():
    mapped_dataframes[name] = apply_column_mapping(dataframe, column_mapping)

if mapped_dataframes:
    print("Mapped order columns:")
    if "olist_orders_dataset" in mapped_dataframes:
        print(list(mapped_dataframes["olist_orders_dataset"].columns))
    logging.info("Applied column mappings to %s DataFrames", len(mapped_dataframes))
else:
    print("No new CSV files to map.")
    logging.info("No new CSV files to map")

2026-11-25 15:12:17,417 INFO: No new CSV files to map


No new CSV files to map.


## 4. Validate rows and save bad records


In [5]:
from pipeline_helpers import (
    MissingRequiredColumnError,
    check_required_columns,
    find_missing_fields,
    get_required_columns,
    make_bad_record,
)

if not mapped_dataframes:
    print("No new files to validate. Existing bad_records.csv was not changed.")
    logging.info("No new files to validate")
else:
    bad_rows = []
    try:
        for name, dataframe in mapped_dataframes.items():
            required_columns = get_required_columns(name, config["required_columns"])
            check_required_columns(name, dataframe.columns, required_columns)
            missing_rows = dataframe[required_columns].isna().any(axis=1)

            for row_number, row in dataframe[missing_rows].iterrows():
                row_values = row.to_dict()
                missing_fields = find_missing_fields(row_values, required_columns)
                bad_rows.append(
                    make_bad_record(f"{name}.csv", row_number + 2, row_values, missing_fields)
                )

    except MissingRequiredColumnError as error:
        print(f"Validation stopped: {error}")
        logging.error("Validation stopped: %s", error)
    else:
        bad_records = pd.DataFrame(bad_rows)
        if bad_records.empty:
            bad_records = pd.DataFrame(columns=["source_file", "row_number", "reason"])

        bad_records_file = project_folder / config["bad_records_file"]
        if manifest_file.is_file() and bad_records_file.is_file():
            previous_bad_records = pd.read_csv(bad_records_file)
            bad_records = pd.concat([previous_bad_records, bad_records], ignore_index=True)

        bad_records.to_csv(bad_records_file, index=False)

        processed_files = save_manifest(
            manifest_file,
            processed_files,
            [file.name for file in files_to_process],
        )

        print(f"Found {len(bad_records)} rows with missing required values.")
        print(f"Bad records saved to {bad_records_file}")
        logging.info("Validation finished: %s bad rows saved", len(bad_records))
        logging.info("Manifest now contains %s processed files", len(processed_files))

2026-11-25 15:12:19,108 INFO: No new files to validate


No new files to validate. Existing bad_records.csv was not changed.


## 5. Use a custom exception and handle the failure

Try a required column that does not exist. The custom exception should give a clear message without stopping the notebook.

In [6]:
try:
    check_required_columns(
        "orders test",
        pd.DataFrame(columns=["order_id"]),
        ["order_id", "missing_column"],
    )
except MissingRequiredColumnError as error:
    print(f"Handled error: {error}")
    logging.warning("Handled demonstration error: %s", error)

2026-11-25 15:12:20,985 WARNING: Handled demonstration error: orders test is missing required columns: missing_column


Handled error: orders test is missing required columns: missing_column


## 6. Write pipeline logs to a file

The earlier cells log CSV loading, mapping, validation, and handled errors. The log folder is set in `day15_config.yaml`.

In [7]:
print(f"Log file: {log_file}")
print(log_file.read_text(encoding="utf-8"))

Log file: c:\Users\Arshathdeen\Desktop\TAP-Python\day-15\logs\run_20261125_151212_895233.log
2026-11-25 15:12:12,897 INFO: Skipping already processed file: olist_customers_dataset.csv
2026-11-25 15:12:12,899 INFO: Skipping already processed file: olist_customers_semicolon.csv
2026-11-25 15:12:12,902 INFO: Skipping already processed file: olist_geolocation_dataset.csv
2026-11-25 15:12:12,903 INFO: Skipping already processed file: olist_order_items_dataset.csv
2026-11-25 15:12:12,904 INFO: Skipping already processed file: olist_order_payments_dataset.csv
2026-11-25 15:12:12,906 INFO: Skipping already processed file: olist_order_reviews_dataset.csv
2026-11-25 15:12:12,907 INFO: Skipping already processed file: olist_orders_dataset.csv
2026-11-25 15:12:12,910 INFO: Skipping already processed file: olist_products_dataset.csv
2026-11-25 15:12:12,911 INFO: Skipping already processed file: olist_sellers_dataset.csv
2026-11-25 15:12:12,912 INFO: Skipping already processed file: product_category

## 7. Skip files already processed

The manifest stores processed CSV filenames. Run the notebook again to confirm there are no new files and the existing bad-record output is left alone.

In [8]:
print(f"Files in manifest: {len(processed_files)}")
print(f"New files processed this run: {len(files_to_process)}")
if not files_to_process:
    print("Nothing new to process.")

Files in manifest: 10
New files processed this run: 0
Nothing new to process.


## 8. Run the unit tests

Run the focused tests for the Day 15 pipeline helpers from the project folder.

In [9]:
!pytest "{day15_folder / 'test_pipeline_helpers.py'}" -q

........                                                                 [100%]
8 passed in 0.53s


## 10. Adapt to a new file with an extra column

In [10]:
extra_file_config = {
    "column_mapping": {
        **config["column_mapping"],
        "reviewer_note": "review_comment",
    },
    "required_columns": {
        "reviewer_feedback": ["order_id", "status", "review_comment"],
    },
}

reviewer_file = pd.DataFrame(
    {
        "order_id": ["order-new"],
        "order_status": ["delivered"],
        "reviewer_note": ["Fast delivery"],
    }
)
mapped_reviewer_file = apply_column_mapping(
    reviewer_file, extra_file_config["column_mapping"]
)
required_columns = get_required_columns(
    "reviewer_feedback", extra_file_config["required_columns"]
)
check_required_columns(
    "reviewer_feedback", mapped_reviewer_file.columns, required_columns
)

print(list(mapped_reviewer_file.columns))
print("The new column was mapped and passed the required-column check.")

['order_id', 'status', 'review_comment']
The new column was mapped and passed the required-column check.
